# NTK Results

## Preliminaries

In [1]:
from jax import config
config.update("jax_enable_x64", True)

In [8]:
import pickle
import os

import jax
import jax.numpy as jnp

from src.utils import *
from src.functions import *
from src.pdes import *
from src.ntk import *

from flax import nnx
import optax

from jaxkan.KAN import KAN

from sklearn.model_selection import train_test_split

results = dict()

## Function Fitting

We first perform experiments relevant to the NTK for the Function Fitting case, because PDEs have their own NTK formulation.

### Parameters

In [9]:
N = 5000
n_ntk = 256

seed = 42

num_epochs = 2001
checkpoints = [0, 500, 1000, 1500, 2000]

opt_type = optax.adam(learning_rate=0.001)

pows = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]

# Model input/output
n_in, n_out = 2, 1

# Studied functions
funcs = [("f3", f3)]

In [10]:
# ------------------------
# Big architecture details
# ------------------------
G_big = 20
hidden_big = [32, 32, 32]

### Helper Functions

In [11]:
def get_data(func, N, n_ntk, seed):
    
    # Generate data
    x, y = generate_func_data(func, 2, N, seed)
    
    # Split data (at this point just to ensure continuity)
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=seed)
    
    # Subsample points used to compute NTK
    key_ntk = jax.random.PRNGKey(seed)
    idx = jax.random.choice(key_ntk, X_train.shape[0], shape=(n_ntk,), replace=False)
    X_ntk = X_train[idx]

    return X_train, y_train, X_ntk

In [12]:
def run_experiment(func, model, opt, X_train, y_train, X_ntk, title):
    spec_list, tau_list = [], []
    conds, ranks = [], []

    # τ = 0 (before any updates)
    K0 = stabilize_kernel(ntk_matrix(model, X_ntk))
    lam0 = jnp.sort(jnp.linalg.eigvalsh(K0))[::-1]
    spec_list.append(lam0)
    tau_list.append(0)
    conds.append(cond_from_eigs(lam0))
    
    eff_rank0 = (lam0.sum() ** 2) / (jnp.sum(lam0 ** 2) + 1e-12)
    ranks.append(float(eff_rank0))

    for epoch in range(num_epochs):
        loss = func_fit_step(model, opt, X_train, y_train)

        if epoch in checkpoints[1:]:
            Kt = stabilize_kernel(ntk_matrix(model, X_ntk))
            lam = jnp.sort(jnp.linalg.eigvalsh(Kt))[::-1]
            spec_list.append(lam)
            tau_list.append(epoch)
            conds.append(cond_from_eigs(lam))
            
            eff_rank_t = (lam.sum() ** 2) / (jnp.sum(lam ** 2) + 1e-12)
            ranks.append(float(eff_rank_t))

    l2error = func_fit_eval(model, func, 2, 200)

    print(f"\t{title} Model Metrics:")
    print(f"\tCond Number: τ=0 → {conds[0]:.2e}, τ={tau_list[-1]} → {conds[-1]:.2e}")
    print(f"\tEffective Rank: τ=0 → {ranks[0]:.2f}, τ={tau_list[-1]} → {ranks[-1]:.2f}")
    print(f"\tFinal Loss = {loss:.2e}\t L^2 Error = {l2error:.2e}\n")

    return spec_list, tau_list, conds, ranks

### Main Routine

In [14]:
results["function"] = dict()

for func_name, func in funcs:

    for pow_basis in pows:
    
        results["function"][pow_basis] = dict()

        for pow_res in pows:
    
            results["function"][pow_basis][pow_res] = dict()

            params_big_power = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

            # Get the data for the function
            X_train, y_train, X_ntk = get_data(func, N, n_ntk, seed)
        
            # Define the big architecture
            layer_dims = [n_in, *hidden_big, n_out]
        
            print(f"Training model with dimensions {layer_dims} for function {func_name}.")
            
            power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_power, seed = seed+1)
            power_opt = nnx.Optimizer(power_model, opt_type)
        
            spec_list, tau_list, conds, ranks = run_experiment(func, power_model, power_opt, X_train, y_train, X_ntk, "Power")
            results["function"][pow_basis][pow_res]["spec_list"] = spec_list
            results["function"][pow_basis][pow_res]["tau_list"] = tau_list
            results["function"][pow_basis][pow_res]["conds"] = conds
            results["function"][pow_basis][pow_res]["ranks"] = ranks

Training model with dimensions [2, 32, 32, 32, 1] for function f3.
	Power Model Metrics:
	Cond Number: τ=0 → 6.33e+03, τ=2000 → 7.79e+04
	Effective Rank: τ=0 → 47.46, τ=2000 → 40.89
	Final Loss = 1.60e+00	 L^2 Error = 1.42e+01

Training model with dimensions [2, 32, 32, 32, 1] for function f3.
	Power Model Metrics:
	Cond Number: τ=0 → 1.32e+04, τ=2000 → 1.83e+05
	Effective Rank: τ=0 → 69.00, τ=2000 → 74.51
	Final Loss = 3.64e-03	 L^2 Error = 6.11e-01

Training model with dimensions [2, 32, 32, 32, 1] for function f3.
	Power Model Metrics:
	Cond Number: τ=0 → 7.72e+02, τ=2000 → 1.29e+03
	Effective Rank: τ=0 → 75.69, τ=2000 → 74.57
	Final Loss = 1.20e-01	 L^2 Error = 6.00e-01

Training model with dimensions [2, 32, 32, 32, 1] for function f3.
	Power Model Metrics:
	Cond Number: τ=0 → 8.00e+02, τ=2000 → 1.39e+03
	Effective Rank: τ=0 → 72.63, τ=2000 → 68.99
	Final Loss = 3.25e-01	 L^2 Error = 6.12e-01

Training model with dimensions [2, 32, 32, 32, 1] for function f3.
	Power Model Metrics:

In [16]:
# Save function results for further processing
with open("ntk_powgrid.pkl", "wb") as f:
    pickle.dump(results, f)

## PDE

In [17]:
# Setup
pdes = [("burgers", burgers_res)]

N = 2**6
n_ntk_pde = 256
n_ntk_bc = 32

RBA_gamma = 0.999
RBA_eta = 0.01

seed = 42

num_epochs = 5001
checkpoints = [0, 1000, 2000, 3000, 4000, 5000]

n_in, n_out = 2, 1

opt_type = optax.adam(learning_rate=0.001)

In [18]:
def get_data(pde_name, N, n_ntk_pde=256, n_ntk_bc=32, seed=42):
    
    # Get the reference solution
    refsol, coords = get_ref(pde_name)

    pde_collocs, bc_collocs, bc_data = get_collocs(pde_name, N)
    
    # consistent NTK subsets per experiment
    key = jax.random.PRNGKey(seed)
    
    idx_pde = jax.random.choice(key, pde_collocs.shape[0], shape=(min(n_ntk_pde, pde_collocs.shape[0]),), replace=False)
    
    idx_bc  = jax.random.choice(key,  bc_collocs.shape[0], shape=(min(n_ntk_bc,  bc_collocs.shape[0]),),  replace=False)

    return refsol, coords, pde_collocs, bc_collocs, bc_data, idx_pde, idx_bc

In [19]:
def run_experiment_pde(pde_res_fn, model, opt, refsol, coords, pde_collocs, bc_collocs, bc_data, idx_pde, idx_bc, title):

    # NTK X, Y
    X_pde_ntk = pde_collocs[idx_pde]
    
    X_bc_ntk  = bc_collocs[idx_bc]
    Y_bc_ntk  = bc_data[idx_bc]

    # init RBA weights for training loop
    l_E = jnp.ones((pde_collocs.shape[0], 1))
    l_B = jnp.ones((bc_collocs.shape[0], 1))

    specE_list, specB_list, tau_list = [], [], []
    condsE, condsB = [], []

    # τ = 0
    wE0 = l_E[idx_pde].ravel()
    wB0 = l_B[idx_bc].ravel()
    lamE0, lamB0 = pinntk_diag_spectra_weighted(model, pde_res, X_pde_ntk, X_bc_ntk, Y_bc_ntk, wE0, wB0)
    
    specE_list.append(lamE0)
    specB_list.append(lamB0)
    tau_list.append(0)
    
    condsE.append(cond_from_eigs(lamE0))
    condsB.append(cond_from_eigs(lamB0))

    for epoch in range(num_epochs):
        loss, l_E, l_B = train_step(model, opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)

        if epoch in checkpoints[1:]:
            wEt = l_E[idx_pde].ravel()
            wBt = l_B[idx_bc].ravel()
            lamE, lamB = pinntk_diag_spectra_weighted(model, pde_res, X_pde_ntk, X_bc_ntk, Y_bc_ntk, wEt, wBt)
            
            specE_list.append(lamE)
            specB_list.append(lamB)
            tau_list.append(epoch)
            
            condsE.append(cond_from_eigs(lamE))
            condsB.append(cond_from_eigs(lamB))

    output = model(coords).reshape(refsol.shape)
    l2error = jnp.linalg.norm(output-refsol)/jnp.linalg.norm(refsol)

    print(f"\t{title} Metrics:")
    print(f"\tPDE Cond#: τ=0 → {condsE[0]:.2e}, τ={tau_list[-1]} → {condsE[-1]:.2e}")
    print(f"\tBC  Cond#: τ=0 → {condsB[0]:.2e}, τ={tau_list[-1]} → {condsB[-1]:.2e}")
    print(f"\tFinal Loss = {loss:.2e}\t L^2 Error = {l2error:.2e}\n")

    return specE_list, specB_list, tau_list, condsE, condsB


In [ ]:
results["pde"] = dict()

for pde_name, pde_res in pdes:

    # Define the loss function for this PDE
    def loss_fn(model, l_E, l_B, pde_collocs, bc_collocs, bc_data):

        # ------------- PDE ---------------------------- #
        pde_residuals = pde_res(model, pde_collocs)
    
        # Get new RBA weights
        abs_pde_res = jnp.abs(pde_residuals)
        l_E_new = (RBA_gamma*l_E) + (RBA_eta*abs_pde_res/jnp.max(abs_pde_res))
    
        # Multiply by RBA weights
        w_resids_pde = l_E_new * pde_residuals
    
        # Get loss
        pde_loss = jnp.mean(w_resids_pde**2)
    
    
        # ------------- BC ----------------------------- #
        bc_residuals = model(bc_collocs) - bc_data
    
        # Get new RBA weights
        abs_bc_res = jnp.abs(bc_residuals)
        l_B_new = (RBA_gamma*l_B) + (RBA_eta*abs_bc_res/jnp.max(abs_bc_res))
    
        # Multiply by RBA weights
        w_resids_bc = l_B_new * bc_residuals
    
        # Loss
        bc_loss = jnp.mean(w_resids_bc**2)
    
        
        # ------------- Total --------------------------- #
        total_loss = pde_loss + bc_loss
    
        return total_loss, (l_E_new, l_B_new)
        
    # Define the train step
    @nnx.jit
    def train_step(model, optimizer, l_E, l_B, pde_collocs, bc_collocs, bc_data):
    
        (loss, (l_E_new, l_B_new)), grads = nnx.value_and_grad(loss_fn, has_aux = True)(model, l_E, l_B, pde_collocs, bc_collocs, bc_data)
    
        optimizer.update(grads)
    
        return loss, l_E_new, l_B_new

    # Get the data
    refsol, coords, pde_collocs, bc_collocs, bc_data, idx_pde, idx_bc = get_data(pde_name, N, n_ntk_pde, n_ntk_bc, seed)

    # Define the big architecture
    layer_dims = [n_in, *hidden_big, n_out]

    print(f"Training model with dimensions {layer_dims} for PDE {pde_name}.")

    for pow_basis in pows:
    
        results["pde"][pow_basis] = dict()

        for pow_res in pows:
    
            results["pde"][pow_basis][pow_res] = dict()

            params_big_power = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}


            
            power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_power, seed = seed+1)
            power_opt = nnx.Optimizer(power_model, opt_type)
        
            specE_list, specB_list, tau_list, condsE, condsB = run_experiment_pde(pde_res, power_model, power_opt, refsol, coords, pde_collocs, 
                                                                                  bc_collocs, bc_data, idx_pde, idx_bc, "Power")
            
            results["pde"][pow_basis][pow_res]["specE_list"] = specE_list
            results["pde"][pow_basis][pow_res]["specB_list"] = specB_list
            results["pde"][pow_basis][pow_res]["tau_list"] = tau_list
            results["pde"][pow_basis][pow_res]["condsE"] = condsE
            results["pde"][pow_basis][pow_res]["condsB"] = condsB

Training model with dimensions [2, 32, 32, 32, 1] for PDE burgers.


In [ ]:
# Save pde results for further processing
with open("ntk_powgrid.pkl", "wb") as f:
    pickle.dump(results, f)